# EEG Cognitive Stress Classification

## Data Loading and Exploration

This script loads EEG recording, checks for problems, and creates visualizations that build intuition about what EEG signals look like and how they differ between stressed vs relaxed states.


DATASET: SAM 40
- 40 subjects, 32 EEG channels, 128 Hz sampling rate
- 4 tasks (Relaxation, Stroop, Arithmetic, Mirror Image)
- 3 trials per task per subject = 480 total recordings
- Each recording = 25 seconds of brain activity

In [1]:
# Imports:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # Using non-interactive backend (works everywhere)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.io import loadmat
from glob import glob
import warnings
warnings.filterwarnings('ignore')
 
# Setting a clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

In [2]:
# Path location:

DATA_DIR = os.path.join(r"C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification", "\Data", "SAM40")
print(DATA_DIR)

OUTPUT_DIR = os.path.join(r"C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification", "\Results", "phase1")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(OUTPUT_DIR)

C:\Data\SAM40
C:\Results\phase1


Constants:

The 32 EEG channel names, in the exact order stored in the .mat files.
These follow the 10-20 international system for electrode placement.

Naming convention:
- Letter = brain region:
    - Fp = Frontopolar (forehead)
    - F  = Frontal
    - FC = Fronto-Central
    - FT = Fronto-Temporal
    - C  = Central
    - T  = Temporal
    - CP = Centro-Parietal
    - P  = Parietal
    - PO = Parieto-Occipital 
    - O  = Occipital
- Number = hemisphere:
    - z  = midline (center)
    - odd numbers = left hemisphere
    - even numbers = right hemisphere

In [3]:


CHANNEL_NAMES = [
    'Cz', 'Fz', 'Fp1', 'F7', 'F3', 'FC1', 'C3', 'FC5',
    'FT9', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO9', 'O1',
    'Pz', 'Oz', 'O2', 'PO10', 'P8', 'P4', 'CP2', 'CP6',
    'T8', 'FT10', 'FC6', 'C4', 'FC2', 'F4', 'F8', 'Fp2'
]
 
SAMPLING_RATE = 128       # Samples per second (Hz)
TRIAL_DURATION = 25       # Each recording is 25 seconds long
EXPECTED_SAMPLES = 3200   # 128 Hz × 25 s = 3200 samples
N_CHANNELS = 32
N_SUBJECTS = 40
N_TRIALS = 3
 
# The 4 experimental tasks
TASK_NAMES = ['Relaxation', 'Stroop', 'Mirror_Image', 'Arithmetic']
 
# Colors for each task (used consistently across all plots)
TASK_COLORS = {
    'Relaxation':   '#2ecc71',   # Green — calm state
    'Stroop':       '#e74c3c',   # Red — high cognitive interference
    'Mirror_Image': '#f39c12',   # Orange — spatial reasoning
    'Arithmetic':   '#3498db',   # Blue — mental computation
}
 
# Self-reported stress ratings from Table 1 of the SAM 40 paper (Data in Brief).
# Format: subject_id -> [[T1_stroop, T1_arith, T1_mirror],
#                         [T2_stroop, T2_arith, T2_mirror],
#                         [T3_stroop, T3_arith, T3_mirror]]
STRESS_RATINGS = {
    1:[[3,6,3],[2,7,5],[4,4,7]], 2:[[5,3,4],[4,3,4],[3,7,5]],
    3:[[4,5,3],[5,3,5],[5,8,7]], 4:[[4,5,3],[2,3,5],[5,7,5]],
    5:[[6,6,6],[2,5,3],[3,5,7]], 6:[[2,5,4],[3,4,3],[5,8,6]],
    7:[[4,5,5],[3,3,3],[6,6,3]], 8:[[4,3,5],[3,6,6],[3,5,6]],
    9:[[4,3,6],[4,4,4],[6,7,4]], 10:[[4,3,4],[3,4,5],[5,6,4]],
    11:[[4,7,3],[5,6,5],[5,5,6]], 12:[[2,5,7],[5,5,4],[3,8,6]],
    13:[[3,3,5],[2,3,4],[1,3,5]], 14:[[3,3,4],[2,6,4],[4,6,3]],
    15:[[5,9,6],[3,7,5],[5,6,5]], 16:[[1,4,4],[1,5,6],[6,8,5]],
    17:[[8,8,9],[4,6,8],[6,7,7]], 18:[[6,6,5],[5,5,6],[7,9,6]],
    19:[[6,4,6],[3,2,4],[3,4,1]], 20:[[7,10,9],[5,10,9],[6,10,8]],
    21:[[9,7,8],[9,8,8],[9,7,8]], 22:[[5,9,8],[6,8,8],[5,7,6]],
    23:[[1,3,2],[1,4,2],[5,8,4]], 24:[[1,4,2],[1,2,1],[4,5,7]],
    25:[[2,2,1],[2,2,1],[5,5,7]], 26:[[8,6,7],[7,6,8],[6,4,5]],
    27:[[3,5,5],[2,3,4],[3,5,3]], 28:[[7,7,8],[5,7,7],[4,6,5]],
    29:[[6,6,7],[6,8,7],[5,6,6]], 30:[[5,7,6],[5,8,6],[4,7,5]],
    31:[[4,10,8],[3,9,7],[7,6,3]], 32:[[1,5,5],[1,1,2],[3,7,6]],
    33:[[3,5,4],[2,3,2],[6,8,3]], 34:[[1,2,3],[1,3,1],[3,2,2]],
    35:[[1,6,5],[1,5,2],[5,4,6]], 36:[[6,4,4],[3,5,4],[5,5,5]],
    37:[[4,6,5],[2,5,3],[6,8,4]], 38:[[5,6,4],[3,5,6],[3,4,5]],
    39:[[3,6,4],[3,5,5],[5,6,3]], 40:[[5,3,5],[5,4,6],[3,4,4]],
}

## Understanding .MAT File Format

How .MAT File work?

A .mat file is MATLAB's way of saving variables. When we load it in Python using scipy.io.loadmat(), we get a Python dictionary where:
- Keys starting with '__' are MATLAB metadata (ignore these)
- Other keys are the actual data variables
    
For the SAM 40 dataset, the EEG data is stored as a 2D array:
- Rows = channels (32)
- Columns = time samples (~3200)
    
The data is in EEGLAB .set format, which is just a .mat file with specific variable names.


In [4]:

"""
Loading one .mat file and return the EEG data as a numpy array

Parameters:
    filepath: path to a .mat file
    
Returns:
    eeg_data: numpy array of shape (32, ~3200)
    data_key: the key name used to store the EEG data
"""

def load_single_mat(filepath):
    mat_contents = loadmat(filepath)
    
    # Filter out MATLAB metadata keys (starting with '__')
    data_keys = [k for k in mat_contents.keys() if not k.startswith('__')]
    
    # Find the largest array (EEG data)
    eeg_data = None
    data_key = None
    for key in data_keys:
        arr = mat_contents[key]
        if isinstance(arr, np.ndarray):
            # Handle nested MATLAB structures (EEGLAB format)
            if arr.dtype == np.object_:             # If MATLAB structures are loaded as object arrays in Numpy
                try:
                    inner = arr.flat[0]
                    if hasattr(inner, 'dtype') and inner.dtype.names:     # To check if it's a structured array (MATLAB struct) and check for a field named 'data'
                        if 'data' in inner.dtype.names:
                            arr = inner['data'].flat[0]
                except (IndexError, AttributeError):
                    continue
            
            if eeg_data is None or arr.size > (eeg_data.size if eeg_data is not None else 0):
                eeg_data = arr.astype(np.float64)         # Converts largest array to 64 bit floating point for numerical processing
                data_key = key      # Stores the name of the variable that holds the largest array (EEG data)
    
    return eeg_data, data_key




In [5]:


"""

    Extracting subject number, task name, and trial number from filename.
    
    Naming convention in the dataset:
        Relax_sub_1_trial1.mat             → Relaxation, Subject 1, Trial 1
        Stroop_sub_5_trial2.mat            → Stroop, Subject 5, Trial 2
        Mirror_image_sub_10_trial3.mat     → Mirror Image, Subject 10, Trial 3
        Arithmetic_sub_40_trial1.mat       → Arithmetic, Subject 40, Trial 1
    
    Pattern: Task_sub_N_trialM.mat
    
    Returns: subject_id (int), task_name (str), trial_num (int)
"""

def parse_filename(filename):
    name = filename.replace('.mat', '')
    parts = name.split('_')
    
    # Extract subject number (after 'sub')
    subject_id = int(parts[parts.index('sub') + 1])
    
    # Extract trial number - attached after trial like "trial1", "trial2"
    # Find the part that starts with "trial" and extract the number from it
    trial_part = [p for p in parts if p.startswith('trial')][0]
    trial_num = int(trial_part.replace('trial', ''))
    
    # Identify task from keywords in the filename
    fname_lower = filename.lower()
    if 'relax' in fname_lower:
        task_name = 'Relaxation'
    elif 'stroop' in fname_lower:
        task_name = 'Stroop'
    elif 'mirror' in fname_lower:
        task_name = 'Mirror_Image'
    elif 'arithmetic' in fname_lower:
        task_name = 'Arithmetic'
    else:
        task_name = 'Unknown'
    
    return subject_id, task_name, trial_num

## Loading Data

Loading every .mat file from the filtered_data folder into a structured dictionary.

The dataset provides both raw and filtered versions. The filtered data has already been cleaned by the researchers:
1. Band-pass filtered (0.5-45 Hz) to remove electrical noise
2. Baseline drift removed using Savitzky-Golay filter
3. Artifacts (eye blinks, muscle movements) removed using wavelet thresholding
    
Using the pre-filtered data lets us focus on feature extraction and classification without worrying about complex preprocessing.

In [6]:

def load_all_data(data_dir):
    # Returns:
    #     data_dict: {subject_id: {task_name: {trial_num: np.array(32, ~3200)}}}
    #     file_info: list of dicts with metadata about each loaded file

    filtered_dir = os.path.join(data_dir, 'filtered_data')
    if not os.path.exists(filtered_dir):
        print(f"ERROR: {filtered_dir} not found!")
        print(f"Make sure you extracted the dataset correctly.")
        return None, None
    
    mat_files = sorted(glob(os.path.join(filtered_dir, '*.mat')))
    print(f"Found {len(mat_files)} .mat files in filtered_data/")
    
    data_dict = {}
    file_info = []
    errors = []
    
    for i, filepath in enumerate(mat_files):
        filename = os.path.basename(filepath)
        
        # Show progress every 50 files
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  Loading file {i+1}/{len(mat_files)}: {filename}")
        
        try:
            subject_id, task_name, trial_num = parse_filename(filename)
            eeg_data, data_key = load_single_mat(filepath)
            
            if eeg_data is None:
                errors.append(f"No data found in {filename}")
                continue
            
            # Store in nested dictionary
            if subject_id not in data_dict:
                data_dict[subject_id] = {}
            if task_name not in data_dict[subject_id]:
                data_dict[subject_id][task_name] = {}
            data_dict[subject_id][task_name][trial_num] = eeg_data
            
            # Record file metadata
            file_info.append({
                'filename': filename,
                'subject': subject_id,
                'task': task_name,
                'trial': trial_num,
                'n_channels': eeg_data.shape[0],
                'n_samples': eeg_data.shape[1] if eeg_data.ndim > 1 else len(eeg_data),
                'data_key': data_key,
                'min': eeg_data.min(),
                'max': eeg_data.max(),
                'mean': eeg_data.mean(),
                'std': eeg_data.std(),
            })
            
        except Exception as e:
            errors.append(f"{filename}: {str(e)}")
    
    file_info_df = pd.DataFrame(file_info)
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"LOADING COMPLETE")
    print(f"{'='*60}")
    print(f"Subjects loaded:  {len(data_dict)}")
    print(f"Files loaded:     {len(file_info)}")
    print(f"Files expected:   {N_SUBJECTS * len(TASK_NAMES) * N_TRIALS}")
    
    if errors:
        print(f"\nErrors ({len(errors)}):")
        for e in errors[:5]:
            print(f"  {e}")
    
    return data_dict, file_info_df